# Week 1 — EDA Kickoff (Dong Dong)

**Goal:** Quick stats and core charts on cleaned flights (after BTS cleaning).

**Input:** `data/processed/bts/flights_2024_clean.parquet` (or auto-download via Step 0)

**Outputs:** Summary prints + figures saved under `data/reports/eda/`

## Step 0: Ensure project data (local or Hugging Face)

Pulls [ahiruuu/CIS-5450](https://huggingface.co/datasets/ahiruuu/CIS-5450) if `data/` is not ready. Then Step 1 loads the cleaned parquet.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
for _p in [_here] + list(_here.parents):
    if (_p / "notebooks" / "project_data.py").exists():
        sys.path.insert(0, str(_p / "notebooks"))
        break

from project_data import ensure_project_data

ensure_project_data()

## Step 1: Load data (local / venv / Colab)

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import seaborn as sns

from project_data import resolve_project_root

plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid')

PROJECT_ROOT = resolve_project_root()
DATA_ROOT = Path(os.getenv('FLIGHT_DATA_DIR', PROJECT_ROOT / 'data')).expanduser().resolve()
FIG_DIR = DATA_ROOT / 'reports' / 'eda'
FIG_DIR.mkdir(parents=True, exist_ok=True)

clean_path = DATA_ROOT / 'processed' / 'bts' / 'flights_2024_clean.parquet'
df = pd.read_parquet(clean_path)
df['FlightDate'] = pd.to_datetime(df['FlightDate'])
df['hour'] = df['CRSDepTime'] // 100

print(f'Loaded: {len(df):,} rows, {len(df.columns)} columns from {clean_path}')
df.head(3)

## Step 2：总体延误统计

In [ ]:
total = len(df)
delayed = df['DepDel15'].sum()
on_time = total - delayed

print('=== 总体延误情况 ===')
print(f'总航班数:        {total:,}')
print(f'准点 (<15min):   {on_time:,} ({on_time/total*100:.1f}%)')
print(f'延误 (≥15min):   {delayed:,} ({delayed/total*100:.1f}%)')
print()
print('=== 延误时间分布 ===')
print(df['DepDelay'].describe(percentiles=[.5, .75, .90, .95, .99]))

## Step 3：图1 — 延误率随小时变化（出发时段）

In [ ]:
hourly = df.groupby('hour').agg(
    delay_rate=('DepDel15', 'mean'),
    flight_count=('DepDel15', 'count')
).reset_index()

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

ax1.bar(hourly['hour'], hourly['flight_count'], color='steelblue', alpha=0.4, label='Flight count')
ax2.plot(hourly['hour'], hourly['delay_rate'] * 100, color='crimson', marker='o', linewidth=2, label='Delay rate')

ax1.set_xlabel('Scheduled Departure Hour')
ax1.set_ylabel('Number of Flights', color='steelblue')
ax2.set_ylabel('Delay Rate (%)', color='crimson')
ax2.yaxis.set_major_formatter(mtick.PercentFormatter())

ax1.set_xticks(range(0, 24))
plt.title('Flight Count and Delay Rate by Departure Hour (2024)')
fig.legend(loc='upper left', bbox_to_anchor=(0.1, 0.9))
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig1_hourly_delay.png', dpi=150)
plt.show()
print('关键发现：填写你观察到的规律，例如：晚间航班（18-22点）延误率最高，早晨6-8点延误率最低')

## Step 4：图2 — 延误率随星期变化

In [ ]:
df['day_of_week'] = df['FlightDate'].dt.dayofweek
dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

weekly = df.groupby('day_of_week')['DepDel15'].mean().reset_index()
weekly['label'] = weekly['day_of_week'].map(dict(enumerate(dow_labels)))

plt.figure(figsize=(8, 4))
bars = plt.bar(weekly['label'], weekly['DepDel15'] * 100,
               color=['#d62728' if r > weekly['DepDel15'].mean() else '#1f77b4'
                      for r in weekly['DepDel15']])
plt.axhline(weekly['DepDel15'].mean() * 100, color='gray', linestyle='--', label='Average')
plt.ylabel('Delay Rate (%)')
plt.title('Delay Rate by Day of Week (2024)')
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig2_weekly_delay.png', dpi=150)
plt.show()

## Step 5：图3 — 延误率随月份变化（季节效应）

In [ ]:
df['month'] = df['FlightDate'].dt.month
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

monthly = df.groupby('month').agg(
    delay_rate=('DepDel15', 'mean'),
    avg_delay=('DepDelay', 'mean')
).reset_index()
monthly['label'] = monthly['month'].map(dict(enumerate(month_labels, 1)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(monthly['label'], monthly['delay_rate'] * 100, marker='o', color='crimson')
ax1.set_title('Delay Rate by Month')
ax1.set_ylabel('Delay Rate (%)')
ax1.tick_params(axis='x', rotation=45)

ax2.plot(monthly['label'], monthly['avg_delay'], marker='s', color='steelblue')
ax2.set_title('Average Delay (minutes) by Month')
ax2.set_ylabel('Average Delay (min)')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig3_monthly_delay.png', dpi=150)
plt.show()

## Step 6：图4 — 各航空公司延误率对比

In [ ]:
airline_stats = df.groupby('Reporting_Airline').agg(
    delay_rate=('DepDel15', 'mean'),
    avg_delay_min=('DepDelay', 'mean'),
    flight_count=('DepDel15', 'count')
).reset_index()

# 只看航班量超过 1000 的航空公司
airline_stats = airline_stats[airline_stats['flight_count'] > 1000].sort_values('delay_rate', ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#d62728' if r > airline_stats['delay_rate'].mean() else '#1f77b4'
          for r in airline_stats['delay_rate']]
ax.barh(airline_stats['Reporting_Airline'], airline_stats['delay_rate'] * 100, color=colors)
ax.axvline(airline_stats['delay_rate'].mean() * 100, color='gray', linestyle='--', label='Average')
ax.set_xlabel('Delay Rate (%)')
ax.set_title('Delay Rate by Airline (2024)')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig4_airline_delay.png', dpi=150)
plt.show()

print(airline_stats[['Reporting_Airline','delay_rate','avg_delay_min','flight_count']]
      .to_string(index=False))

## Step 7：图5 — 延误原因构成

In [ ]:
# 只看实际延误的航班
delayed_flights = df[df['DepDel15'] == 1]

cause_cols = ['CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay']
cause_labels = ['Carrier', 'Weather', 'NAS (ATC)', 'Security', 'Late Aircraft']

cause_totals = delayed_flights[cause_cols].sum()
cause_totals.index = cause_labels

plt.figure(figsize=(8, 5))
wedges, texts, autotexts = plt.pie(
    cause_totals,
    labels=cause_labels,
    autopct='%1.1f%%',
    colors=sns.color_palette('Set2', len(cause_cols))
)
plt.title('Breakdown of Delay Causes (Delayed Flights Only, 2024)')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig5_delay_causes.png', dpi=150)
plt.show()

print(cause_totals.sort_values(ascending=False))

## Week 1 EDA 小结

（在这里写下你的观察，提交前完善）

- **时段**：
- **星期**：
- **季节**：
- **航空公司**：
- **延误原因**：

**Week 2 计划深入分析**：
- 地理热力图（各机场延误率）
- 天气与延误的关系（weather join 完成后）
- 连锁延误效应分析（tail_number 跟踪）
- 假设检验（4.11-4.13）